# CardioIA — Fase 4 · Parte 1: Pré-processamento de imagens de ECG

> ⚠️ **AVISO CLÍNICO:** Este sistema é um protótipo acadêmico e **NÃO** substitui avaliação médica.  

**Objetivo:** organizar o dataset Mendeley completo (~928 ECGs, 4 classes), aplicar redimensionamento/normalização e criar splits estratificados treino/validação/teste (`seed=42`).

**Fonte:** DOI [10.17632/gwbz3fsgp8.2](http://dx.doi.org/10.17632/gwbz3fsgp8.2) · Licença CC BY 4.0

## 0. Ambiente (Colab ou local)

In [ ]:
import os
import sys
from pathlib import Path

EM_COLAB = False
try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    pass

if EM_COLAB:
    # Ajuste se o repositório foi clonado/enviado para outro caminho no Drive
    REPO = Path('/content/cardioAI')
    if not REPO.exists():
        REPO = Path('/content')
    %pip install -q opencv-python-headless scikit-learn pandas matplotlib tqdm requests
else:
    # notebook em fase4/notebooks/ → sobe dois níveis até a raiz do repo
    REPO = Path.cwd()
    if (REPO / 'pipeline_imagens.py').exists():
        REPO = REPO.parent  # se cwd = fase4/src
    if (REPO / 'src' / 'pipeline_imagens.py').exists():
        FASE4 = REPO
    else:
        # procura pasta fase4 a partir do cwd
        cand = Path.cwd().resolve()
        FASE4 = None
        for p in [cand, *cand.parents]:
            if (p / 'fase4' / 'src' / 'pipeline_imagens.py').exists():
                REPO = p
                FASE4 = p / 'fase4'
                break
            if (p / 'src' / 'pipeline_imagens.py').exists() and p.name == 'fase4':
                FASE4 = p
                REPO = p.parent
                break
        if FASE4 is None:
            raise FileNotFoundError('Não achei fase4/src/pipeline_imagens.py — rode a partir do repo CardioIA.')

if EM_COLAB:
    FASE4 = REPO / 'fase4' if (REPO / 'fase4').exists() else REPO

sys.path.insert(0, str(FASE4 / 'src'))
print('REPO =', REPO)
print('FASE4 =', FASE4)
print('Colab =', EM_COLAB)

## 1. Download do dataset completo (se necessário)

In [ ]:
ORIGEM = FASE4 / 'data' / 'imagens_ecg_completo'
DESTINO = FASE4 / 'data' / 'processado'

n_jpg = sum(1 for _ in ORIGEM.rglob('*.jpg')) if ORIGEM.exists() else 0
print(f'Imagens já presentes: {n_jpg}')

if n_jpg < 900:
    script = FASE4 / 'data' / 'baixar_ecg_mendeley.py'
    # importa e executa a função de download
    import importlib.util
    spec = importlib.util.spec_from_file_location('baixar_ecg_mendeley', script)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.baixar_dataset(ORIGEM)
else:
    print('Dataset completo já disponível — pulando download.')

## 2. Inventário e exemplos visuais

In [ ]:
import matplotlib.pyplot as plt
import cv2
from collections import Counter
from pipeline_imagens import CLASSES, listar_imagens, SEED, TAMANHO_ALVO

df = listar_imagens(ORIGEM)
print('Total:', len(df))
print(df['classe'].value_counts().reindex(CLASSES))

# dimensão de uma amostra por classe
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, classe in zip(axes, CLASSES):
    caminho = df[df['classe'] == classe].iloc[0]['caminho']
    bgr = cv2.imread(caminho)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(f'{classe}\n{rgb.shape[1]}×{rgb.shape[0]}')
    ax.axis('off')
plt.suptitle('Exemplos brutos (laudo ECG 12 derivações)')
plt.tight_layout()
plt.show()

## 3. Pipeline: resize 224×224, normalização [0,1], split 70/15/15

In [ ]:
from pipeline_imagens import ConfigPipeline, materializar_processado, carregar_e_preprocessar

cfg = ConfigPipeline(origem=ORIGEM, destino=DESTINO)
manifesto = materializar_processado(cfg)
resumo = manifesto.groupby(['particao', 'classe']).size().unstack(fill_value=0)
resumo = resumo.reindex(index=['treino', 'validacao', 'teste'], columns=list(CLASSES))
display(resumo)
print('Seed =', SEED, '| tamanho alvo =', TAMANHO_ALVO)

In [ ]:
# Comparação visual: bruto vs processado
amostra = manifesto[manifesto['classe'] == 'Normal'].iloc[0]
bruto = cv2.cvtColor(cv2.imread(amostra['origem']), cv2.COLOR_BGR2RGB)
proc = carregar_e_preprocessar(amostra['origem'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bruto)
axes[0].set_title(f'Bruto {bruto.shape[1]}×{bruto.shape[0]}')
axes[0].axis('off')
axes[1].imshow(proc)
axes[1].set_title(f'Processado {proc.shape[1]}×{proc.shape[0]} (float [0,1])')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print('Manifesto CSV:', DESTINO / 'manifesto_split.csv')
print('Manifesto JSON:', DESTINO / 'manifesto_split.json')

## 4. Data augmentation (apenas treino — demonstração)

Política adotada: rotação leve (±8°), zoom (±10%) e variação de brilho.  
**Sem flip horizontal** — o laudo de 12 derivações não é espelhável sem distorcer a leitura espacial do ECG.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

gen = ImageDataGenerator(
    rotation_range=8,
    zoom_range=0.1,
    brightness_range=(0.9, 1.1),
    fill_mode='nearest',
)

base = (proc * 255).astype('uint8')
batch = np.expand_dims(base, 0)
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(base)
axes[0].set_title('Original')
axes[0].axis('off')
it = gen.flow(batch, batch_size=1, seed=SEED)
for i in range(4):
    aug = next(it)[0].astype('uint8')
    axes[i + 1].imshow(aug)
    axes[i + 1].set_title(f'Aug {i + 1}')
    axes[i + 1].axis('off')
plt.suptitle('Augmentation só no treino (exemplos)')
plt.tight_layout()
plt.show()

## 5. Conclusão da Parte 1

- Dataset completo baixado e organizado por classe.
- Imagens em RGB, 224×224, normalizadas.
- Split estratificado 70/15/15 com `random_state=42` (manifesto versionável em CSV/JSON).
- Augmentation documentada; justificação clínica/técnica no relatório `docs/relatorio_preprocessamento.md`.

Próximo passo: notebook `02_cnn_classificacao.ipynb`.